# 02 — The point-in-time feature table

A look inside `data/processed/features.parquet` (run `uv run python -m octagon build-features` first).

Everything here was computable *before* each fight happened — that's the whole game.
`uv run python -m octagon audit` verifies it; this notebook is for building intuition.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from octagon.config import resolve_path

df = pd.read_parquet(resolve_path("data/processed/features.parquet"))
diff_cols = sorted(c for c in df.columns if c.startswith("diff_"))
print(f"{len(df)} fights x {len(df.columns)} columns ({len(diff_cols)} differentials)")
df[["date", "f1_name", "f2_name", "label", "swapped", "diff_elo", "diff_age"]].tail(5)

In [ ]:
# Feature availability grows over time: early-era fights lack stats,
# debuting fighters lack everything. Models must cope with NaN.
df["year"] = df["date"].str[:4].astype(int)
availability = df.groupby("year")[["diff_elo", "diff_age", "diff_reach_cm",
                                    "diff_slpm_l3", "diff_ctrl_pct_l3"]].apply(
    lambda g: g.notna().mean())
availability.plot(figsize=(10, 4), title="share of fights where each differential is known")
plt.ylabel("available")
plt.show()

In [ ]:
# Single-feature signal check: does each differential separate winners at all?
# (Point-biserial-ish: mean of diff for label=1 minus label=0, standardized.)
rows = []
for c in diff_cols:
    s = df[c].astype(float)
    if s.notna().sum() < 500 or s.std() == 0:
        continue
    z = (s - s.mean()) / s.std()
    rows.append({"feature": c, "standardized gap": z[df.label == 1].mean() - z[df.label == 0].mean()})
signal = pd.DataFrame(rows).set_index("feature").sort_values("standardized gap")
signal.plot.barh(figsize=(8, 10), legend=False,
                 title="winner-vs-loser gap per differential (larger |x| = more univariate signal)")
plt.tight_layout(); plt.show()

In [ ]:
# The Elo landscape: distribution of pre-fight Elo differences and how often
# the higher-rated fighter actually wins, per bucket.
elo = df[df["diff_elo"].notna()].copy()
elo["bucket"] = pd.cut(elo["diff_elo"], bins=range(-400, 401, 100))
obs = elo.groupby("bucket", observed=True)["label"].agg(["mean", "size"])
obs.columns = ["observed f1 win rate", "n fights"]
obs

## Takeaways

- Rolling-stat differentials only become widely available ~2010+; the model's real training signal is concentrated in the modern era.
- Elo difference is the strongest univariate signal, with age difference close behind — matching both the literature and the logistic regression coefficients in `reports/evaluation_valid.md`.
- The Elo bucket table doubles as a calibration check on Elo itself: observed win rates should rise smoothly with rating difference.